# Finance Strategy Expert — RAG Lab (Phase 1 prototype)

Interactive notebook to build and test the RAG pipeline **before** extracting modules.

**Stack (no extra signups, ~zero cost):**
- **OpenAI** for embeddings (`text-embedding-3-small`) and generation (`gpt-4o-mini`) — uses your existing OpenAI key.
- **Qdrant on-disk** (`QdrantClient(path="data/qdrant")`) — persistent, no Docker needed. Survives kernel restarts; only new/changed docs are re-embedded.
- Corpus: `data/corpus/*.md` finance docs.

**Grounding rule:** answer ONLY from retrieved corpus chunks; if nothing passes the similarity threshold, reply *"I don't know."*

## 1. Install dependencies
Run once. Safe to skip if already installed. Uses **uv** (installs into this kernel's interpreter).

In [6]:
import sys
# !uv add install -q --python {sys.executable} openai qdrant-client python-dotenv

## 2. Configuration & OpenAI key
Loads `.env` if present; otherwise prompts you to paste the key (not stored to disk).

In [ ]:
import os, glob, getpass, hashlib, json, uuid
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (sk-...): ")

# Models
EMBED_MODEL = "text-embedding-3-small"   # 1536-dim, very cheap
CHAT_MODEL  = "gpt-4o-mini"              # cheap, good for grounded Q&A
VECTOR_SIZE = 1536

# Tunables (tweak these live in section 8)
TOP_K           = 5
SCORE_THRESHOLD = 0.30     # cosine; start permissive, tune later
CHUNK_WORDS     = 250

COLLECTION = "finance_kb"

# Locate the corpus whether the notebook runs from repo root or notebooks/
CORPUS_DIR = Path("data/corpus")
if not CORPUS_DIR.exists():
    CORPUS_DIR = Path("../data/corpus")
assert CORPUS_DIR.exists(), f"corpus not found at {CORPUS_DIR.resolve()}"

# Persistent vector store lives next to the corpus (data/qdrant/), resolved the
# same way as CORPUS_DIR so it works from repo root or notebooks/. Gitignored.
QDRANT_DIR = CORPUS_DIR.parent / "qdrant"
QDRANT_DIR.mkdir(parents=True, exist_ok=True)

# Fixed namespace -> deterministic uuid5 point IDs (stable per doc+chunk index).
NAMESPACE = uuid.UUID("d3b07384-d9a0-4c9b-8e2a-00000000feed")

IDK_MESSAGE = "I don't know — I couldn't find that in my knowledge base."
# Grounding is enforced in TWO places: (1) the retrieval threshold gate, which
# short-circuits to IDK_MESSAGE when nothing relevant is found (the LLM is never
# called); (2) this prompt. The prompt PERMITS synthesis/comparison across the
# retrieved sources — otherwise the model refuses multi-part questions even when
# the context supports them — while still forbidding outside knowledge.
SYSTEM_PROMPT = (
    "You are a strategy-investment expert. Answer using ONLY the information in "
    "the context below — never outside knowledge. You may synthesize, compare, "
    "and summarize across the provided sources to answer. For a comparison, "
    "cover each item the context supports; if the context covers some items but "
    "not others, answer for those it does cover and explicitly say which it "
    'lacks. Only if the context is essentially irrelevant to the question, reply '
    'exactly "I don\'t know." Cite the source title for each claim.'
)
print("Corpus:", CORPUS_DIR.resolve())
print("Files:", len(list(CORPUS_DIR.glob("*.md"))))
print("Store:", QDRANT_DIR.resolve())

## 3. Chunker — frontmatter parsing + word-bounded splitting

In [ ]:
@dataclass
class Chunk:
    text: str
    title: str
    source_url: str
    doc_id: str = ""
    score: float | None = None


def parse_frontmatter(raw: str):
    """Return (meta dict, body) from a markdown doc with --- frontmatter."""
    if raw.startswith("---"):
        end = raw.index("\n---", 3)
        fm_block = raw[3:end].strip()
        body = raw[end + 4:].strip()
        meta = {}
        for line in fm_block.splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                meta[k.strip()] = v.strip().strip('"')
        return meta, body
    return {}, raw.strip()


def chunk_text(body: str, size: int):
    words = body.split()
    if not words:
        return []
    return [" ".join(words[i:i + size]) for i in range(0, len(words), size)]


def chunk_document(raw: str, size: int, doc_id: str = ""):
    meta, body = parse_frontmatter(raw)
    title = meta.get("title", "")
    url = meta.get("source_url", "")
    return [Chunk(text=t, title=title, source_url=url, doc_id=doc_id)
            for t in chunk_text(body, size)]


# Load + chunk the whole corpus (preview only; ingest() re-reads per doc)
all_chunks = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    all_chunks.extend(chunk_document(path.read_text(encoding="utf-8"), CHUNK_WORDS, path.name))

print(f"{len(all_chunks)} chunks from {len(list(CORPUS_DIR.glob('*.md')))} docs")
print("Example chunk:\n", all_chunks[0].title, "|", all_chunks[0].source_url)
print(all_chunks[0].text[:300], "...")

## 4. Embeddings (OpenAI)

In [ ]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env

# Token-usage accumulator. Cumulative across all OpenAI calls since the
# kernel started (or since the last reset_token_usage()).
TOKENS = {"embed": 0, "chat_in": 0, "chat_out": 0}

def reset_token_usage():
    """Zero the counters — call before a run you want to measure cleanly."""
    for k in TOKENS:
        TOKENS[k] = 0


def embed(texts, _model=EMBED_MODEL):
    """Return a list of embedding vectors for a list of strings."""
    resp = client.embeddings.create(model=_model, input=texts)
    TOKENS["embed"] += resp.usage.total_tokens
    return [d.embedding for d in resp.data]


# quick sanity check
_v = embed(["hello world"])
print("vector dim:", len(_v[0]))

## 5. Ingest into persistent Qdrant (incremental)

On-disk store at `data/qdrant/`. `ingest()` embeds only **new or changed** docs (tracked by a sha256 manifest) and skips unchanged ones — so restarts and re-runs don't re-embed the whole corpus.

- **New doc** → added. **Edited doc** → its old points are replaced. **Unchanged** → skipped (no OpenAI call).
- Run `ingest(force=True)` to wipe and rebuild — required after changing `EMBED_MODEL`, `VECTOR_SIZE`, or `CHUNK_WORDS`.
- Deleting a doc from the corpus leaves its old chunks until the next `force=True` rebuild.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue,
)

# Persistent, on-disk Qdrant — survives kernel restarts (no re-embed on reload).
# Re-running this cell in the same kernel: close the old client first so the
# directory lock is released, then reopen.
if "qdrant" in globals():
    try:
        qdrant.close()
    except Exception:
        pass
qdrant = QdrantClient(path=str(QDRANT_DIR))

MANIFEST_PATH = QDRANT_DIR / "manifest.json"


def load_manifest():
    if MANIFEST_PATH.exists():
        try:
            return json.loads(MANIFEST_PATH.read_text())
        except json.JSONDecodeError:
            return {}
    return {}


def save_manifest(m):
    MANIFEST_PATH.write_text(json.dumps(m, indent=2))


def ensure_collection():
    if not qdrant.collection_exists(COLLECTION):
        qdrant.create_collection(
            collection_name=COLLECTION,
            vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
        )


def _point_id(doc_id, idx):
    return str(uuid.uuid5(NAMESPACE, f"{doc_id}:{idx}"))


def _delete_doc_points(doc_id):
    qdrant.delete(
        collection_name=COLLECTION,
        points_selector=Filter(must=[
            FieldCondition(key="doc_id", match=MatchValue(value=doc_id))
        ]),
    )


def ingest(force=False):
    """Incremental ingest: embed only new/changed docs, skip unchanged ones.

    - New doc        -> embed + add (additive).
    - Changed doc    -> delete its old points by doc_id, then add new chunks.
    - Unchanged doc  -> skipped, no OpenAI call.
    - force=True     -> wipe collection + manifest and rebuild everything
                        (required after changing EMBED_MODEL, VECTOR_SIZE,
                        or CHUNK_WORDS).
    """
    if force and qdrant.collection_exists(COLLECTION):
        qdrant.delete_collection(COLLECTION)
    ensure_collection()
    manifest = {} if force else load_manifest()

    embedded_docs, skipped_docs, added_points = 0, 0, 0
    for path in sorted(CORPUS_DIR.glob("*.md")):
        doc_id = path.name
        raw = path.read_text(encoding="utf-8")
        h = hashlib.sha256(raw.encode("utf-8")).hexdigest()

        if manifest.get(doc_id) == h:
            skipped_docs += 1
            continue

        if doc_id in manifest:          # changed doc -> per-doc replace
            _delete_doc_points(doc_id)

        chunks = chunk_document(raw, CHUNK_WORDS, doc_id)
        if chunks:
            vectors = embed([c.text for c in chunks])
            points = [
                PointStruct(
                    id=_point_id(doc_id, i), vector=vec,
                    payload={"text": c.text, "title": c.title,
                             "source_url": c.source_url, "doc_id": doc_id},
                )
                for i, (c, vec) in enumerate(zip(chunks, vectors))
            ]
            qdrant.upsert(collection_name=COLLECTION, points=points)
            added_points += len(points)

        manifest[doc_id] = h
        embedded_docs += 1

    save_manifest(manifest)
    total = qdrant.count(collection_name=COLLECTION).count
    print(f"ingest: embedded {embedded_docs} doc(s), skipped {skipped_docs}, "
          f"+{added_points} point(s); collection now holds {total}.")
    return {"embedded_docs": embedded_docs, "skipped_docs": skipped_docs,
            "total_points": total}


ingest()

## 6. Retriever — search + similarity threshold (the grounding gate)

Two retrievers:
- **`search()`** — plain single-vector top-K (used by `debug_search` and section-9 tuning).
- **`search_multi()`** — expands the question into focused sub-queries (one per item in a comparison), retrieves for each, and unions the results. This gives **balanced** context for comparison/multi-part questions, where a single embedding otherwise collapses onto one document. `answer()` uses this one.

Both apply the same `SCORE_THRESHOLD` gate, so off-topic questions still retrieve nothing and short-circuit to "I don't know."

In [ ]:
def search(question, top_k=None, threshold=None):
    """Plain single-vector retrieval (used by debug_search and section-9 tuning)."""
    top_k = TOP_K if top_k is None else top_k
    threshold = SCORE_THRESHOLD if threshold is None else threshold
    qvec = embed([question])[0]
    hits = qdrant.query_points(collection_name=COLLECTION, query=qvec, limit=top_k).points
    kept = []
    for h in hits:
        if h.score >= threshold:
            p = h.payload
            kept.append(Chunk(text=p["text"], title=p["title"], source_url=p["source_url"],
                              doc_id=p.get("doc_id", ""), score=h.score))
    return kept


def expand_queries(question, n=4):
    """Break a question into focused retrieval sub-queries so multi-part and
    comparison questions retrieve BALANCED context across every item involved.

    A single embedding of "compare A, B and C" collapses onto whichever doc is
    most similar overall, so top-K fills with chunks from just that one doc. One
    sub-query per compared item fixes that. Always includes the original
    question; returns a deduped list.
    """
    resp = client.chat.completions.create(
        model=CHAT_MODEL, temperature=0,
        messages=[
            {"role": "system", "content":
                f"Rewrite the user's question into up to {n} focused search queries that "
                "together retrieve everything needed to answer it. For a comparison, write "
                "one query per item being compared. Output ONLY the queries, one per line, "
                "with no numbering or bullets."},
            {"role": "user", "content": question},
        ],
    )
    TOKENS["chat_in"]  += resp.usage.prompt_tokens
    TOKENS["chat_out"] += resp.usage.completion_tokens
    subs = [ln.strip(" -•\t").strip() for ln in resp.choices[0].message.content.splitlines()]
    return list(dict.fromkeys([question] + [s for s in subs if s]))


def search_multi(question, per_query_k=4, threshold=None):
    """Coverage-oriented retrieval: expand into sub-queries, retrieve for each,
    union the hits (dedup by point id, keeping the best score). This is what
    answer() uses so comparison/aggregation questions get balanced context.
    """
    threshold = SCORE_THRESHOLD if threshold is None else threshold
    queries = expand_queries(question)
    best = {}
    for qvec in embed(queries):                      # one batched embeddings call
        for h in qdrant.query_points(collection_name=COLLECTION, query=qvec, limit=per_query_k).points:
            if h.score >= threshold and (h.id not in best or h.score > best[h.id].score):
                p = h.payload
                best[h.id] = Chunk(text=p["text"], title=p["title"], source_url=p["source_url"],
                                   doc_id=p.get("doc_id", ""), score=h.score)
    return sorted(best.values(), key=lambda c: c.score, reverse=True)


def debug_search(question, top_k=None):
    top_k = TOP_K if top_k is None else top_k
    qvec = embed([question])[0]
    hits = qdrant.query_points(collection_name=COLLECTION, query=qvec, limit=top_k).points
    print(f"Q: {question}\n")
    for h in hits:
        mark = "PASS" if h.score >= SCORE_THRESHOLD else "drop"
        print(f"  [{mark}] {h.score:.3f}  {h.payload['title']}")
        print(f"         {h.payload['text'][:120]}...\n")

debug_search("What is EV/EBIT?")

## 7. Answerer — grounded generation or "I don't know"

Short-circuits to *I don't know* when nothing passes the threshold (the LLM is never called). When context **is** found, the prompt lets the model **synthesize and compare across sources** (so multi-part questions get answered) while still forbidding outside knowledge and requiring citations. For a partial comparison it answers what the context supports and names what's missing.

In [ ]:
@dataclass
class Answer:
    text: str
    sources: list
    found: bool


def build_user_prompt(question, chunks):
    context = "\n\n".join(
        f"[Source: {c.title} | {c.source_url}]\n{c.text}" for c in chunks
    )
    return f"Context:\n{context}\n\nQuestion: {question}"


def answer(question):
    chunks = search_multi(question)          # coverage retrieval (handles comparisons)
    if not chunks:
        return Answer(text=IDK_MESSAGE, sources=[], found=False)
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(question, chunks)},
        ],
    )
    TOKENS["chat_in"]  += resp.usage.prompt_tokens
    TOKENS["chat_out"] += resp.usage.completion_tokens
    text = resp.choices[0].message.content
    sources = list(dict.fromkeys(f"{c.title} — {c.source_url}" for c in chunks))
    return Answer(text=text, sources=sources, found=True)


def ask(question):
    """Pretty-print an end-to-end answer."""
    r = answer(question)
    print(f"Q: {question}\n")
    print(r.text)
    if r.found and r.sources:
        print("\nSources:")
        for s in r.sources:
            print("  -", s)
    print("\n" + "-" * 60)
    return r

## 8. Test it 🎯

**In-corpus questions** (should answer with citations):

In [ ]:
ask("What is EV/EBIT and why do value investors use it?")
ask("What is the Acquirer's Multiple?")
ask("What did Tobias Carlisle find about mechanical value investing in Deep Value?")
ask("What is O'Shaughnessy's composite value factor?")

**Out-of-corpus question** (should say *I don't know*):

In [ ]:
ask("What's the weather in Tel Aviv today?")
ask("Who won the 2018 World Cup?")

## 9. Tune the grounding threshold
Use `debug_search` to see scores, then adjust `SCORE_THRESHOLD` and re-`ask`.
- Too many "I don't know" on valid questions → **lower** the threshold.
- Off-topic questions getting answered → **raise** it.
Once happy, copy `SCORE_THRESHOLD`, `TOP_K`, `CHUNK_WORDS` into the modules later.

In [ ]:
# Example: see raw scores for a borderline query, then experiment
debug_search("How does enterprise value differ from market cap?")

# Try a stricter threshold live:
# SCORE_THRESHOLD = 0.40
# ask("What's the weather in Tel Aviv today?")   # should now reliably say I don't know

## 10. Token usage & cost
`TOKENS` accumulates across every OpenAI call since the kernel started (or since the last `reset_token_usage()`). To measure a single clean run: call `reset_token_usage()`, re-run the ingest + test cells, then run the report below.

In [ ]:
# Prices in USD per 1M tokens — update if OpenAI changes them.
PRICE_PER_1M = {
    "embed":    0.02,   # text-embedding-3-small
    "chat_in":  0.15,   # gpt-4o-mini input
    "chat_out": 0.60,   # gpt-4o-mini output
}

def report_token_usage():
    total = sum(TOKENS.values())
    cost = sum(TOKENS[k] * PRICE_PER_1M[k] / 1_000_000 for k in TOKENS)
    print("Token usage (cumulative since kernel start / last reset):")
    print(f"  embeddings  : {TOKENS['embed']:>9,} tok")
    print(f"  chat input  : {TOKENS['chat_in']:>9,} tok")
    print(f"  chat output : {TOKENS['chat_out']:>9,} tok")
    print(f"  {'-'*30}")
    print(f"  TOTAL       : {total:>9,} tok")
    print(f"  est. cost   : ${cost:.5f}")

report_token_usage()